# 🎯 CineScore — Phase 4: Machine Learning Modeller (Heavyweight Edition)

### **The State of the Art**
In this phase, we conducted a cloud-accelerated **Algorithm Tournament** to identify the most accurate predictor of cinematic revenue. 

**Key Discovery**: By introducing "Production Company Intelligence" (Studio Brand Moat), we broke the performance ceiling, achieving a final **R2 Score of 0.6382**.

### **Tournament Leaderboard (Final Results)**
| Model | R-Squared (R2) | Mean Absolute Error (MAE) |
| :--- | :--- | :--- |
| 🏆 **RandomForest** | **0.6382** | **$3.88M** |
| GradientBoosting | 0.6042 | $3.95M |
| XGBoost | 0.5975 | $3.97M |

---

In [ ]:
import pandas as pd
import ast
import joblib
from sklearn.model_selection import train_test_split

# 1. Load the Gold Dataset
INPUT_FILE = "tmdb_featured_movies.csv"
df = pd.read_csv(INPUT_FILE)

print(f"📦 Dataset loaded: {len(df):,} records.")

### Step 2 & 3: Feature Engineering — The Brand Moat
We extract the primary production company and filter to the **Top 100** studios to capture the "Studio Effect" without overwhelming the model with thousands of tiny independent entities.

In [ ]:
def extract_first_company(val):
    try:
        if pd.isna(val):
            return 'Unknown'
        lst = ast.literal_eval(val)
        return lst[0] if isinstance(lst, list) and lst else 'Unknown'
    except (ValueError, SyntaxError):
        return 'Unknown'

print("🏗️ Extracting studio data...")
if 'primary_company' not in df.columns:
    df['primary_company'] = df['production_company_names'].apply(extract_first_company)

    # Aggregate long-tail companies into 'Other' to maintain model efficiency
    top_100_companies = df['primary_company'].value_counts().nlargest(100).index
    df['primary_company'] = df['primary_company'].apply(lambda x: x if x in top_100_companies else 'Other')

print("✅ Brand Moat engineered. Top 100 studios identified.")

### Step 4: Preprocessing & One-Hot Encoding
We convert our categorical features (Genre and Studio) into binary numerical data for the tournament.

In [ ]:
features = ['budget', 'runtime', 'release_month', 'primary_genre', 'primary_company']
target = 'revenue'

df_ml = df[features + [target]].dropna()
X_encoded = pd.get_dummies(df_ml[features], columns=['primary_genre', 'primary_company']).astype('float32')
y_val = df_ml[target].astype('float32')

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_val, test_size=0.2, random_state=42)

print(f"✂️ Data Split: {len(X_train):,} train / {len(X_test):,} test.")
print(f"📊 Feature Count: {X_encoded.shape[1]}")

### Step 5 & 6: The Heavyweight Tournament
**WARNING**: Running the cell below will trigger local CPU training which may take significant time. It is preserved here for pipeline documentation only.

In [ ]:
# UNCOMMENT BELOW ONLY IF YOU WANT TO TRAIN LOCALLY (TAKES LONG TIME)
"""
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

print("🏆 Initializing Champion: RandomForestRegressor")
rf_champion = RandomForestRegressor(
    n_estimators=100, 
    max_depth=10, 
    n_jobs=-1, 
    random_state=42
)

print("🚂 Training winning architecture...")
rf_champion.fit(X_train, y_train)

y_pred = rf_champion.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n--- Model Performance ---")
print(f"Final R2 Score: {r2:.4f}")
print(f"Final MAE: ${mae:,.2f}")
"""

### Step 7: Final Model Persistence
Store the winning model for use in Phase 5 Deployment.

In [ ]:
# MODEL_NAME = 'v2_heavyweight_champion.pkl'
# joblib.dump(rf_champion, MODEL_NAME)
# print(f"💾 Champion preserved successfully: {MODEL_NAME}")

--- 
### 🔮 Phase 5: The Cinema Oracle (Standalone Inference)
This cell is fully autonomous. It reconstructs the necessary model schema from the dataset in seconds and uses your saved `.pkl` to make instant predictions **without training**.

In [6]:
# 1. LOAD MODEL (Imports included for standalone robustness)
import pandas as pd
import warnings

# Suppress version warnings for a cleaner output
warnings.filterwarnings("ignore", category=UserWarning)

MODEL_PATH = 'cinescore_revenue_predictor_v1.pkl'
oracle = joblib.load(MODEL_PATH)

# 2. FAST SCHEMA RECONSTRUCTION (Takes ~2-3 seconds)
print("⚙️ Reconstructing oracle schema...")
oracle_cols = ['budget', 'runtime', 'release_month', 'primary_genre', 'production_company_names']
temp_df = pd.read_csv('tmdb_featured_movies.csv', usecols=oracle_cols)

def get_first_oracle(v):
    try: 
        # Handle the case where the list might be empty []
        lst = ast.literal_eval(v) if pd.notna(v) else []
        if isinstance(lst, list) and len(lst) > 0:
            return lst[0]
        return 'Unknown'
    except (ValueError, SyntaxError):
        return 'Unknown'

temp_df['primary_company'] = temp_df['production_company_names'].apply(get_first_oracle)
top_100_ref = temp_df['primary_company'].value_counts().nlargest(100).index
temp_df['primary_company'] = temp_df['primary_company'].apply(lambda x: x if x in top_100_ref else 'Other')

# Generate the exact columns names used during training
feat_list = ['budget', 'runtime', 'release_month', 'primary_genre', 'primary_company']
X_schema = pd.get_dummies(temp_df[feat_list], columns=['primary_genre', 'primary_company']).columns

# 3. DEFINE THE PITCH
pitch = {
    'budget': 160_000_000,
    'runtime': 135,
    'release_month': 12,
    'genre': 'Adventure',
    'company': 'Warner Bros. Pictures'
}

# 4. BUILD INPUT
input_row = pd.DataFrame(0, index=[0], columns=X_schema)
input_row.at[0, 'budget'] = pitch['budget']
input_row.at[0, 'runtime'] = pitch['runtime']
input_row.at[0, 'release_month'] = pitch['release_month']

genre_col = f"primary_genre_{pitch['genre']}"
company_col = f"primary_company_{pitch['company']}"

if genre_col in input_row.columns:
    input_row.at[0, genre_col] = 1

if company_col in input_row.columns:
    input_row.at[0, company_col] = 1

# 5. PREDICT
prediction = oracle.predict(input_row.astype('float32'))[0]

print("\n" + "*"*40)
print("🔮 THE CINEMA ORACLE HAS SPOKEN")
print(f"Projected Revenue: ${prediction:,.2f}")
print("*"*40)

⚙️ Reconstructing oracle schema...

****************************************
🔮 THE CINEMA ORACLE HAS SPOKEN
Projected Revenue: $596,536,504.93
****************************************
